# Stockwise — 01: Build Features (Colab)

Thin orchestration notebook per CLAUDE.md Section 9. All logic lives in `src/features/`, already covered by `tests/test_leakage.py`, `tests/test_ingest.py`, `tests/test_pipeline.py`.

**Before running this notebook**, the repo must be pushed to GitHub at `github.com/Kewal-07/retail-demand-forecasting`, since Cell 1 clones from there — Colab cannot see your local machine. Push it, then run cells top to bottom.

Cell 2 needs your own Kaggle account and API token (`kaggle.json`, from kaggle.com/settings). That is the one manual, human-only step.

## Cell 1 — clone or pull, then install

In [ ]:
import os
if not os.path.exists('retail-demand-forecasting'):
    !git clone https://github.com/Kewal-07/retail-demand-forecasting
    %cd retail-demand-forecasting
else:
    %cd retail-demand-forecasting
    !git pull
!pip install -q -r requirements-features.txt

## Cell 2 — Kaggle download (manual: needs your own kaggle.json)

In [ ]:
!pip install -q kaggle
from google.colab import files
files.upload()   # upload kaggle.json here, needs your personal Kaggle credentials
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle competitions download -c m5-forecasting-accuracy -p /content/m5
!unzip -q /content/m5/*.zip -d /content/m5

## Cell 3 — reshape and merge

In [ ]:
from src.features.ingest import load_raw, reshape_and_downcast, merge_sources

raw = load_raw('/content/m5')          # keys: 'sales', 'calendar', 'prices'
sales_long = reshape_and_downcast(raw['sales'])
print(sales_long.memory_usage(deep=True).sum() / 1e9, "GB after downcast")
merged = merge_sources(sales_long, raw['calendar'], raw['prices'])
print(merged.shape)                    # expect ~59.2M rows
del sales_long
import gc
gc.collect()

## Cell 4 — checkpoint to Drive immediately after the merge

Colab disconnects after ~90 min idle; losing the merge means redoing the slowest step.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
merged.to_parquet('/content/drive/MyDrive/m5/merged.parquet')

## Cell 5 — feature engineering, ONE STORE AT A TIME

59.2M rows with the full feature set is roughly 9.5 GB against a 12.7 GB Colab ceiling; calling `build_features(merged)` directly will OOM. Loop per store instead — every feature is computed within a single item-store series, so per-store processing is exactly equivalent to processing the whole frame.

`build_features()` returns exactly the Section 8 feature columns — no label, no row identifiers — so it satisfies `tests/test_pipeline.py::test_feature_set_matches_spec` exactly. That means `sales`, `id`, and `date` have to be reattached here (aligned by `features.index`, since `build_features()` drops rows before an item's first listing) before saving, otherwise the parquet has no label and Cell 6's real-data leakage check below has nothing to check against.

In [ ]:
import gc, os
import pandas as pd
from src.features.pipeline import build_features

os.makedirs('/content/drive/MyDrive/m5/full', exist_ok=True)

ID_AND_LABEL_COLS = ['id', 'd', 'date', 'sales']

for store in merged['store_id'].cat.categories:
    chunk = merged[merged['store_id'] == store].copy()
    features = build_features(chunk)
    identifiers = chunk.loc[features.index, ID_AND_LABEL_COLS]
    out = pd.concat([identifiers, features], axis=1)

    out.to_parquet(f'/content/drive/MyDrive/m5/full/store_id={store}/'
                    'part.parquet', compression='snappy', index=False)
    print(store, out.shape,
          out.memory_usage(deep=True).sum() / 1e9, "GB")
    if store == 'CA_1':
        out.to_parquet('/content/ca1_features.parquet',
                        compression='snappy', index=False)
    del chunk, features, out
    gc.collect()

## Cell 6 — verify no leakage on REAL data, not just the synthetic frame

`tests/test_leakage.py` proves the logic on a synthetic frame. It does not prove the pipeline behaved correctly on real data with thousands of interleaved series per store, which is where a groupby mistake would actually surface. Do both.

In [ ]:
!python -m pytest tests/test_leakage.py -q          # synthetic logic

# real-data check: lag_28 must equal this series' own sales 28 days back
import pandas as pd
ca1 = pd.read_parquet('/content/ca1_features.parquet')
one = ca1[ca1['id'] == ca1['id'].iloc[0]].sort_values('d')
expected = one['sales'].shift(28)
mismatch = (one['lag_28'] - expected).abs().dropna().max()
print("max lag_28 mismatch on real series:", mismatch)
assert mismatch == 0, "lag_28 does not match this series' own history"

## Cell 7 — download the CA_1 sample

Goes into `data/ca1_features.parquet` in the repo. The full 10-store set already went to Drive inside the Cell 5 loop; it is never downloaded.

In [ ]:
from google.colab import files
files.download('/content/ca1_features.parquet')

## Cell 8 — EDA, on a sample, not the full frame

Goes into `reports/eda.html` in the repo. Sampled from CA_1 rather than all stores because the full frame is no longer in memory by this point.

In [ ]:
!pip install -q ydata-profiling
from ydata_profiling import ProfileReport
from google.colab import files

sample = ca1.sample(200_000, random_state=42)
ProfileReport(sample, minimal=True).to_file('/content/eda.html')
files.download('/content/eda.html')

## After Cells 7 and 8

Add both downloaded files to the repo normally from your Mac: `data/ca1_features.parquet` and `reports/eda.html`, then `git add`, `commit`, `push`. Colab never pushes to git directly, keeping credentials off the notebook.